In [ ]:
pip install rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 12.3 MB/s eta 0:00:00


In [ ]:
import openai
import random
import pandas as pd
from tqdm import tqdm
import numpy as np
from sklearn.metrics import f1_score,accuracy_score
from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem import DataStructs
from rdkit.Chem import rdMolDescriptors
from rdkit import Chem
import warnings
from rdkit import RDLogger
import datetime
import os
import time

In [ ]:
!git clone https://github.com/ChemFoundationModels/ChemLLMBench.git

Cloning into 'ChemLLMBench'...
remote: Enumerating objects: 244, done.
remote: Counting objects: 100% (157/157), done.
remote: Compressing objects: 100% (108/108), done.
remote: Total 244 (delta 78), reused 96 (delta 43), pack-reused 87 (from 1)
Receiving objects: 100% (244/244), 4.27 MiB | 17.48 MiB/s, done.
Resolving deltas: 100% (103/103), done.


In [ ]:
!pip install -U bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 13.4 MB/s eta 0:00:00


In [ ]:
import bitsandbytes
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model_name = "meta-llama/Llama-3.2-3B-Instruct"  # or any HF model you can access

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)

model = prepare_model_for_kbit_training(model)

lora_cfg = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.0,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
)

model = get_peft_model(model, lora_cfg)


tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/1.46G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [ ]:
import pandas as pd
from datasets import Dataset

# Path to your training CSV
train_csv = "/content/ChemLLMBench/data/property_prediction/BACE.csv"  # adjust path

df = pd.read_csv(train_csv)

# Detect SMILES column name safely
smiles_col = "mol" # Explicitly set to 'mol' as per DataFrame contents
label_col = "Class"

In [ ]:
def make_prompt(row):
  if int(row[label_col])==1:
    label = "Yes"
  else:
    label = "No"
  return (
        "Instruction:\n"
        "You are an expert chemist specializing in molecular property prediction.\n"
        "Given a molecule’s SMILES string, determine whether the compound can inhibit Beta-site Amyloid Precursor Protein Cleaving Enzyme 1 (BACE1).\n"
        "Base your judgment on structural features such as molecular weight, atom composition, bond types, functional groups, and overall drug-likeness relevant to Alzheimer’s disease therapeutics.\n"
        "Respond with only one word — Yes if the molecule can inhibit BACE1, or No if it cannot.\n"
        "Do not provide any explanation or additional text."
        f"Input:{row[smiles_col]}\n"
        f"Output:{label}"
    )

df["text"] = df.apply(make_prompt, axis=1)


In [ ]:
print(df['text'][0])

Instruction:
You are an expert chemist specializing in molecular property prediction.
Given a molecule’s SMILES string, determine whether the compound can inhibit Beta-site Amyloid Precursor Protein Cleaving Enzyme 1 (BACE1).
Base your judgment on structural features such as molecular weight, atom composition, bond types, functional groups, and overall drug-likeness relevant to Alzheimer’s disease therapeutics.
Respond with only one word — Yes if the molecule can inhibit BACE1, or No if it cannot.
Do not provide any explanation or additional text.Input:O1CC[C@@H](NC(=O)[C@@H](Cc2cc3cc(ccc3nc2N)-c2ccccc2C)C)CC1(C)C
Output:Yes


In [ ]:
df

,mol,CID,Class,Model,pIC50,MW,AlogP,HBA,HBD,RB,...,PEOE7 (PEOE7),PEOE8 (PEOE8),PEOE9 (PEOE9),PEOE10 (PEOE10),PEOE11 (PEOE11),PEOE12 (PEOE12),PEOE13 (PEOE13),PEOE14 (PEOE14),canvasUID,text
0,O1CC[C@@H](NC(=O)[C@@H](Cc2cc3cc(ccc3nc2N)-c2c...,BACE_1,1,Train,9.154901,431.56979,4.4014,3,2,5,...,78.640335,226.855410,107.434910,37.133846,0.000000,7.980170,0.000000,0.000000,1,Instruction:\nYou are an expert chemist specia...
1,Fc1cc(cc(F)c1)C[C@H](NC(=O)[C@@H](N1CC[C@](NC(...,BACE_2,1,Train,8.853872,657.81073,2.6412,5,4,16,...,47.171600,365.676940,174.076750,34.923889,7.980170,24.148668,0.000000,24.663788,2,Instruction:\nYou are an expert chemist specia...
2,S1(=O)(=O)N(c2cc(cc3c2n(cc3CC)CC1)C(=O)N[C@H](...,BACE_3,1,Train,8.698970,591.74091,2.5499,4,3,11,...,47.941147,192.406520,255.752550,23.654478,0.230159,15.879790,0.000000,24.663788,3,Instruction:\nYou are an expert chemist specia...
3,S1(=O)(=O)C[C@@H](Cc2cc(O[C@H](COCC)C(F)(F)F)c...,BACE_4,1,Train,8.698970,591.67828,3.1680,4,3,12,...,37.954151,194.353040,202.763350,36.498634,0.980913,8.188327,0.000000,26.385181,4,Instruction:\nYou are an expert chemist specia...
4,S1(=O)(=O)N(c2cc(cc3c2n(cc3CC)CC1)C(=O)N[C@H](...,BACE_5,1,Train,8.698970,629.71283,3.5086,3,3,11,...,39.361153,179.712880,220.461300,23.654478,0.230159,15.879790,0.000000,26.100143,5,Instruction:\nYou are an expert chemist specia...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1508,Clc1cc2nc(n(c2cc1)C(CC(=O)NCC1CCOCC1)CC)N,BACE_1543,0,Test,3.000000,364.86969,2.5942,3,2,6,...,37.681076,180.226410,95.670128,30.107586,9.368159,7.980170,0.000000,0.000000,1543,Instruction:\nYou are an expert chemist specia...
1509,Clc1cc2nc(n(c2cc1)C(CC(=O)NCc1ncccc1)CC)N,BACE_1544,0,Test,3.000000,357.83731,2.8229,3,2,6,...,47.349350,122.401500,99.877144,30.107586,9.368159,7.980170,0.000000,0.000000,1544,Instruction:\nYou are an expert chemist specia...
1510,Brc1cc(ccc1)C1CC1C=1N=C(N)N(C)C(=O)C=1,BACE_1545,0,Test,2.953115,320.18451,3.0895,2,1,2,...,22.563574,96.290794,58.798935,20.071724,9.368159,0.000000,6.904104,0.000000,1545,Instruction:\nYou are an expert chemist specia...
1511,O=C1N(C)C(=NC(=C1)C1CC1c1cc(ccc1)-c1ccccc1)N,BACE_1546,0,Test,2.733298,317.38440,3.8595,2,1,3,...,9.316234,95.907784,112.609720,20.071724,9.368159,0.000000,6.904104,0.000000,1546,Instruction:\nYou are an expert chemist specia...


In [ ]:
ds_train = Dataset.from_pandas(df[["text"]])


In [ ]:
print(ds_train[0])

{'text': 'Instruction:\nYou are an expert chemist specializing in molecular property prediction.\nGiven a molecule’s SMILES string, determine whether the compound can inhibit Beta-site Amyloid Precursor Protein Cleaving Enzyme 1 (BACE1).\nBase your judgment on structural features such as molecular weight, atom composition, bond types, functional groups, and overall drug-likeness relevant to Alzheimer’s disease therapeutics.\nRespond with only one word — Yes if the molecule can inhibit BACE1, or No if it cannot.\nDo not provide any explanation or additional text.Input:O1CC[C@@H](NC(=O)[C@@H](Cc2cc3cc(ccc3nc2N)-c2ccccc2C)C)CC1(C)C\nOutput:Yes'}


In [ ]:
def tokenize_fn(examples):
    tokens = tokenizer(
        examples["text"],
        truncation=True,
        max_length=512,
        padding="max_length",
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

ds_train_tok = ds_train.map(
    tokenize_fn,
    batched=True,
    remove_columns=ds_train.column_names,
)


Map:   0%|          | 0/1513 [00:00<?, ? examples/s]

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

training_args = TrainingArguments(
    output_dir="bace_out",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=32,
    learning_rate=2e-4,
    num_train_epochs=1,
    fp16=True,
    logging_steps=10,
    save_steps=200,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds_train_tok,
    data_collator=collator,
)

trainer.train()


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,1.654900
20,0.496000
30,0.423600
40,0.365500


TrainOutput(global_step=48, training_loss=0.6701150586207708, metrics={'train_runtime': 1485.8715, 'train_samples_per_second': 1.018, 'train_steps_per_second': 0.032, 'total_flos': 1.315786210148352e+16, 'train_loss': 0.6701150586207708, 'epoch': 1.0})

In [ ]:
model.save_pretrained("bace_lora_adapters")
tokenizer.save_pretrained("bace_lora_adapters")


('bace_lora_adapters/tokenizer_config.json',
 'bace_lora_adapters/special_tokens_map.json',
 'bace_lora_adapters/chat_template.jinja',
 'bace_lora_adapters/tokenizer.json')

In [ ]:
!zip -r /content/bace_lora_adapters.zip /content/bace_lora_adapters

  adding: content/bace_lora_adapters/ (stored 0%)
  adding: content/bace_lora_adapters/tokenizer.json (deflated 85%)
  adding: content/bace_lora_adapters/special_tokens_map.json (deflated 63%)
  adding: content/bace_lora_adapters/tokenizer_config.json (deflated 96%)
  adding: content/bace_lora_adapters/adapter_config.json (deflated 58%)
  adding: content/bace_lora_adapters/chat_template.jinja (deflated 71%)
  adding: content/bace_lora_adapters/adapter_model.safetensors (deflated 8%)
  adding: content/bace_lora_adapters/README.md (deflated 65%)


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer # Already imported, but good for standalone
from peft import PeftModel # Ensure PeftModel is available for merge_and_unload

# The 'model' in the current kernel state is already a PeftModel after training.
# We need to merge the adapters to the base model for inference.
# Ensure the model is in evaluation mode
model.eval()

# Merge the LORA adapters with the base model.
# This will unload the adapters and return a fully merged model.
# The 'model' variable will now hold the merged model.
model = model.merge_and_unload()

# Ensure tokenizer padding_side is set to 'left' for inference
tokenizer.padding_side = "left"

/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:397: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


In [ ]:
# Example inference using a sample SMILES string
test_smiles = pd.read_csv("/content/ChemLLMBench/data/property_prediction/BACE_test.csv")
X_test, Y_test = test_smiles['mol'],test_smiles['Class']


In [ ]:
def create_bace_prompt(input_smiles,pp_examples):
    prompt = "### Instruction:\n"
    "You are an expert chemist specializing in molecular property prediction.\n"
    "Given a molecule’s SMILES string, determine whether the compound can inhibit Beta-site Amyloid Precursor Protein Cleaving Enzyme 1 (BACE1).\n"
    "Base your judgment on structural features such as molecular weight, atom composition, bond types, functional groups, and overall drug-likeness relevant to Alzheimer’s disease therapeutics.\n"
    "Respond with only one word — Yes if the molecule can inhibit BACE1, or No if it cannot.\n"
    "Do not provide any explanation or additional text."
    for example in pp_examples:
        prompt += f"SMILES: {example[0]}\nBACE-1 Inhibit: {example[-1]}\n"
    prompt += f"SMILES: {input_smiles}\nBACE-1 Inhibit:\n"
    return prompt

In [ ]:
def generate_output(prompt):
  inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

  # Generate response
  # max_new_tokens should be small since the expected output is 'Yes' or 'No'
  outputs = model.generate(
      input_ids=inputs["input_ids"],
      attention_mask=inputs["attention_mask"],
      max_new_tokens=5, # 'Yes' or 'No' plus potential newline/token
      do_sample=True,
      top_p=0.9, # To encourage more deterministic output for classification
      temperature=0.7, # Lower temperature for more focused output
      pad_token_id=tokenizer.pad_token_id
      )

  # Decode the generated output
  generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
  # print(f"Full generated text:\n{generated_text}")

  # Extract just the predicted answer (e.g., 'Yes' or 'No')
  try:
    predicted_answer = generated_text.split("### Output:")[-1].strip()
    if "Yes" in predicted_answer:
      predicted_answer = "Yes"
    elif "No" in predicted_answer:
      predicted_answer = "No"
    else:
      # Fallback if the output format is not strictly "Yes" or "No" alone
        predicted_answer = predicted_answer.split('\n')[0].strip()

    return predicted_answer
  except:
    return "Error"

In [ ]:
random.seed(42)
#read bace dataset
bace = pd.read_csv("/content/ChemLLMBench/data/property_prediction/BACE.csv")
sample_size = 100
bace_sample= bace.sample(sample_size)
bace.drop(bace_sample.index, inplace = True)

In [ ]:
# random sampling
def radom_sample_examples(bace,sample_size):
    positive_examples = bace[bace["Class"] == 1].sample(int(sample_size/2))
    negative_examples = bace[bace["Class"] == 0].sample(int(sample_size/2))
    smiles = positive_examples["mol"].tolist() + negative_examples["mol"].tolist()

    class_label = positive_examples["Class"].tolist() + negative_examples["Class"].tolist()
    #convert 1 to "Yes" and 0 to "No"" in class_label
    class_label = ["Yes" if i == 1 else "No" for i in class_label]
    bace_examples = list(zip(smiles, class_label))
    return bace_examples

In [ ]:
# scaffold sampling
def top_k_scaffold_similar_molecules(target_smiles, bace_data, k):
    #drop the target_smiles from the dataset
    bace_data = bace_data[bace_data["mol"] != target_smiles]
    molecule_smiles_list = bace_data['mol'].tolist()
    label_list = bace_data['Class'].tolist()
    label_list = ["Yes" if i == 1 else "No" for i in label_list]

    target_mol = Chem.MolFromSmiles(target_smiles)
    if target_mol is not None:
        target_scaffold = MurckoScaffold.GetScaffoldForMol(target_mol)
    else:
        print("Error: Unable to create a molecule from the provided SMILES string.")
        #drop the target_smiles from the dataset
        return None

    target_scaffold = MurckoScaffold.GetScaffoldForMol(target_mol)
    target_fp = rdMolDescriptors.GetMorganFingerprint(target_scaffold, 2)
    RDLogger.DisableLog('rdApp.warning')
    warnings.filterwarnings("ignore", category=UserWarning)
    similarities = []

    for i,smiles in enumerate(molecule_smiles_list):
        mol = Chem.MolFromSmiles(smiles)
        try:
            scaffold = MurckoScaffold.GetScaffoldForMol(mol)
            scaffold_fp = rdMolDescriptors.GetMorganFingerprint(scaffold, 2)
            tanimoto_similarity = DataStructs.TanimotoSimilarity(target_fp, scaffold_fp)
            # print(tanimoto_similarity)
            similarities.append((smiles, tanimoto_similarity,label_list[i]))
        except:
            continue
    similarities.sort(key=lambda x: x[1], reverse=True)
    top_5_similar_molecules = similarities[:k]
    return top_5_similar_molecules

In [ ]:
mkdir /content/results/

mkdir: cannot create directory ‘/content/results/’: File exists


In [ ]:
models = 'LLAMA'  # keep as provided; ensure your backend routes this to a valid model
sample_num = 8
detail_save_folder = '/content/results/'  # path to save the generated result
paras = 0
sample_method = ['random','scaffold']
file_path = []
for sample_method in sample_method:
  detail_predict_file = detail_save_folder + 'test_{}_{}_{}_{}.csv'.format('bace', models, sample_num, sample_method)
  log_file = detail_save_folder + 'test_{}_{}_{}_{}.log'.format('bace', models, sample_num, sample_method)
  print(detail_predict_file)
  print()

  if os.path.exists(detail_predict_file):
    detail_results = pd.read_csv(detail_predict_file).values.tolist()
  else:
    detail_results = []

  now = datetime.datetime.now()
  date_time_str = now.strftime("%Y-%m-%d %H:%M:%S")
  with open(log_file, "a") as file:
    file.write("=" * 30 + date_time_str + "=" * 30 + "\n")
  file_path.append(detail_predict_file)

  # We now generate exactly ONE prediction per example to respect quotas
  out_cols = ['bace_smiles', 'class_label', 'pred']

/content/results/test_bace_LLAMA_8_random.csv

/content/results/test_bace_LLAMA_8_scaffold.csv



In [ ]:
file_path[1]

'/content/results/test_bace_LLAMA_8_scaffold.csv'

In [ ]:
# random sampling
detail_predict_file = file_path[0]
detail_results = []   # <-- this must be a list, not the file path

para_index = 0
bace_examples = radom_sample_examples(bace, sample_num)

for i in tqdm(range(0, len(bace_sample))):
    if para_index < 0:
        para_index += 1
        continue

    example = [(bace_sample.iloc[i]['mol'], bace_sample.iloc[i]['Class'])]
    pred_y = []
    generated_results = []

    for text in example:
        prompt = create_bace_prompt(text[0], bace_examples)
        with open(log_file, "a") as file:
            file.write(prompt + "\n")
            file.write("=" * 50 + "\n")

        pred_text = generate_output(prompt)
        # time.sleep(21)  # or smaller + exponential backoff if rate limits occur

        generated_results.append(pred_text)
        # Fix: Append the items as a single list, not by concatenating a list with a string.
        detail_results.append([text[0], text[-1], pred_text])

        if (i + 1) % 10 == 0:
            details_df = pd.DataFrame(
                detail_results,
                columns=['bace_smiles', 'class_label', 'pred']
            )
            details_df.to_csv(detail_predict_file, index=False)
            print('save file')

# after loop ends, save final version
details_df = pd.DataFrame(
    detail_results,
    columns=['bace_smiles', 'class_label', 'pred']
)
details_df.to_csv(detail_predict_file, index=False)


 10%|█         | 10/100 [00:12<01:54,  1.27s/it]

save file


 20%|██        | 20/100 [00:27<02:07,  1.59s/it]

save file


 30%|███       | 30/100 [00:38<01:23,  1.19s/it]

...

save file


 90%|█████████ | 90/100 [01:47<00:11,  1.12s/it]

save file


100%|██████████| 100/100 [02:00<00:00,  1.20s/it]

save file


In [ ]:
# scaffold sampling
detail_predict_file = file_path[1]
para_index = 0
for i in tqdm(range(0, len(bace_sample))):
  if para_index < 0:
    para_index += 1
    continue

  example = [(bace_sample.iloc[i]['mol'], bace_sample.iloc[i]['Class'])]

  for text in example:
    bace_examples = top_k_scaffold_similar_molecules(text[0], bace, sample_num)
    prompt = create_bace_prompt(text[0], bace_examples)
    with open(log_file, "a") as file:
      file.write(prompt + "\n")
      file.write("=" * 50 + "\n")

    pred_text = generate_output(prompt)
    # time.sleep(21)  # or smaller + exponential backoff if rate limits occur

    generated_results.append(pred_text)
    detail_results.append([text[0], text[-1], pred_text])

    if (i + 1) % 10 == 0:
      details_df = pd.DataFrame(
        detail_results,
        columns=['bace_smiles', 'class_label', 'pred']
      )
      details_df.to_csv(detail_predict_file, index=False)
      print('save file')

# after loop ends, save final version
details_df = pd.DataFrame(
    detail_results,
    columns=['bace_smiles', 'class_label', 'pred']
)
details_df.to_csv(detail_predict_file, index=False)

 10%|█         | 10/100 [00:36<05:44,  3.83s/it]

save file


 20%|██        | 20/100 [01:23<06:07,  4.60s/it]

save file


 30%|███       | 30/100 [02:02<04:04,  3.49s/it]

...

save file


 90%|█████████ | 90/100 [04:25<00:17,  1.73s/it]

save file


100%|██████████| 100/100 [04:45<00:00,  2.86s/it]

save file


In [ ]:
!zip -r results.zip /content/results

  adding: content/results/ (stored 0%)
  adding: content/results/test_bace_LLAMA_8_random.csv (deflated 78%)
  adding: content/results/test_bace_LLAMA_8_scaffold.csv (deflated 88%)
  adding: content/results/test_bace_LLAMA_8_random.log (deflated 79%)
  adding: content/results/test_bace_LLAMA_8_scaffold.log (deflated 95%)
